In [54]:
### ETL & Pakete
################

import pandas as pd
from sqlalchemy import create_engine
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.express as px
import os
from dotenv import load_dotenv

#.env Datei laden (nur lokal)
dotenv_path = os.path.join(os.getcwd(), '..', '.env')  # eine Ebene hoch
load_dotenv(dotenv_path)

# PostgreSQL-Verbindungsdaten
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
db_name = os.getenv('DB_NAME_GYM')

# DB-Verbindung
connection_string = f'postgresql://{user}:{password}@{host}:5432/{db_name}'
engine = create_engine(
    connection_string, 
    connect_args={"options": "-c client_encoding=utf8"}
)

gym_query = 'SELECT * FROM gym_log'
gym_table = pd.read_sql(gym_query,engine)

gym_table.rename(
    columns={
        'tab_name': 'person',
        'timestamp': 'datum'
    },
    inplace=True
)

In [12]:
### Funktion: Geräte Dropdown mit Update-Logik
##############################################

def update_geraete_options(person, gym, zeitraum):
    start, end = zeitraum
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end) + pd.Timedelta(days=1)
    
    df_filtered = gym_table[
        (gym_table['person'].str.lower() == person.lower()) &
        (gym_table['gym'] == gym) &
        (gym_table['datum'] >= start_dt) &
        (gym_table['datum'] < end_dt)
    ]
    
    geraete_filtered = sorted(df_filtered['gerät'].dropna().unique().tolist())
    
    # Aktuellen Wert sichern, falls er noch gültig ist
    current_value = tuple(geraete_select.value) if geraete_select.value else ()
    new_value = tuple(g for g in current_value if g in geraete_filtered)
    
    # Update Optionen und Wert
    geraete_select.options = geraete_filtered
    geraete_select.value = new_value if new_value else (geraete_filtered[0],) if geraete_filtered else ()


In [47]:
### Funktion: Update Dashboard
##############################

def update_dashboard(person, gym, geraete_sel, zeitraum):
    start, end = zeitraum
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end) + pd.Timedelta(days=1)
    
    df = gym_table[
        (gym_table['person'].str.lower() == person.lower()) &
        (gym_table['gym'] == gym) &
        (gym_table['gerät'].isin(geraete_sel)) &
        (gym_table['datum'] >= start_dt) &
        (gym_table['datum'] < end_dt)
    ].copy()
    
    with summary_output:
        summary_output.clear_output()
        if df.empty:
            display(widgets.HTML("<b>Keine Daten für die gewählten Filter.</b>"))
            return

        gesamt_besuche = df.shape[0]
        geraete_counts = df['gerät'].value_counts().rename_axis("Gerät").reset_index(name="Anzahl")
        left_html = f"""
        <div style="font-family: Arial, sans-serif; line-height: 1.4;">
          <h3>Übersicht für <b>{person}</b> im <b>{gym}</b></h3>
          <p><b>Zeitraum:</b> {start_dt.date()} – {(end_dt - pd.Timedelta(days=1)).date()}</p>
          <p><b>Gesamt-Besuche (Einträge):</b> {gesamt_besuche}</p>
        </div>
        """
        display(widgets.HTML(left_html))

    with content_left:
        content_left.clear_output()
        display(widgets.HTML(geraete_counts.to_html(index=False)))
        
    # Metriken vorbereiten
    metric = metric_selector.value  # 'volumen' | 'gewicht' | 'wdh'

    # Nach Datum und Gerät gruppieren
    df['tag'] = df['datum'].dt.floor('D')
    # Volumen pro Satz
    if metric == 'volumen':
        agg = df.groupby(['tag', 'gerät'])['vol_gesamt'].sum().reset_index()
        y = 'vol_gesamt'
        title = 'Trainingsvolumen pro Gerät über die Zeit'
        y_label = 'Volumen (Gewicht×Wdh)'
    # Durchschnittliches Gewicht
    elif metric == 'gewicht':
        agg = df.groupby(['tag', 'gerät'])['avg_gewicht'].mean().reset_index()
        y = 'avg_gewicht'
        title = 'Mittleres Gewicht je Gerät über die Zeit'
        y_label = 'Gewicht'
    # Durchschnittliche Wiederholungen
    elif metric == 'wdh':
        agg = df.groupby(['tag', 'gerät'])['avg_wdh'].mean().reset_index()
        y = 'avg_wdh'
        title = 'Mittlere Wiederholungen je Gerät über die Zeit'
        y_label = 'Wiederholungen'
    else:
        return

    fig = px.line(
        agg,
        x='tag',
        y=y,
        color='gerät',
        title=title,
        labels={y: y_label, 'tag': 'Datum', 'gerät': 'Gerät'}
    )
    fig.update_layout(legend_title_text='Gerät')

    with content_right:
        content_right.clear_output()
        fig.show()


In [50]:
### Funktion: Callback / Observe-Mechanismus
############################################

# Wenn Person, Gym oder Zeitraum sich ändern, aktualisiere Geräte-Options
def on_change(change):
    if change.owner in [person_dropdown, gym_dropdown, date_range]:
        update_geraete_options(person_dropdown.value, gym_dropdown.value, date_range.value)

    update_dashboard(
        person_dropdown.value,
        gym_dropdown.value,
        geraete_select.value,
        date_range.value
    )

In [44]:
### Vor-Berechnungen und Formatierungen
#######################################

# Umwandlung Datum
gym_table['datum'] = pd.to_datetime(gym_table['datum'], dayfirst=True, errors='coerce')


# Berechne Trainingsvolumen pro Satz und Gesamt
for i in [1, 2, 3]:
    gym_table[f'vol_satz{i}'] = gym_table[f'satz{i}_gew'] * gym_table[f'satz{i}_wdh']


gym_table['vol_gesamt'] = gym_table[[f'vol_satz{i}' for i in [1, 2, 3]]].sum(axis=1)

gym_table['avg_gewicht'] = gym_table[[f'satz{i}_gew' for i in [1, 2, 3]]].mean(axis=1)

gym_table['avg_wdh'] = gym_table[[f'satz{i}_wdh' for i in [1, 2, 3]]].mean(axis=1)


In [53]:
### UI-Controls
###############

### Vorbereitung aller Widgets
##############################
personen = gym_table['person'].dropna().unique().tolist()
personen = sorted(personen, key=lambda x: (x.lower() != 'michi', x.lower()))
person_dropdown = widgets.Dropdown(options=personen, description='Person:', value=personen[0])

gyms = sorted(gym_table['gym'].dropna().unique().tolist())
gym_dropdown = widgets.Dropdown(options=gyms, description='Gym:', value=gyms[0])

geraete = sorted(gym_table['gerät'].dropna().unique().tolist())
geraete_select = widgets.SelectMultiple(options=geraete, description='Geräte:', value=tuple(geraete[:2]))

date_index = pd.date_range(gym_table['datum'].dt.floor('D').min(), 
                           gym_table['datum'].dt.floor('D').max(), 
                           freq='D')

date_options = [(d.strftime('%d.%m.%Y'), d) for d in date_index]

date_range = widgets.SelectionRangeSlider(
    options=date_options,
    index=(0, len(date_options) - 1),
    description='Zeitraum:',
    orientation='horizontal',
    layout=widgets.Layout(width='60%'),
    style={'description_width': 'initial'}
)


metric_selector = widgets.ToggleButtons(
    options=[
        ('Trainingsvolumen', 'volumen'),
        ('Mittleres Gewicht', 'gewicht'),
        ('Mittlere Wiederholungen', 'wdh')
    ],
    description='Metrik:',
    button_style='',
    tooltips=[
        'Summe Gewicht×Wdh über Sätze',
        'Durchschnittliches Gewicht über die 3 Sätze',
        'Durchschnittliche Wiederholungen über die 3 Sätze'
    ]
)


# Befüllung in der Funktion selbst
summary_output = widgets.Output()
summary_output.layout = widgets.Layout(width='50%', min_width='300px')




# Übersicht, welche Widgets können in UI genutzt werden:
# person_dropdown
# gym_dropdown
# geraete_select
# date_range
# metric_selector
# summary_output (= Text-Info)




### Zeile 1: Person & Gym & Zeitraum Slider
###########################################
controls_top = widgets.HBox([
    widgets.Box([person_dropdown], layout=widgets.Layout(width='25%')),
    widgets.Box([gym_dropdown], layout=widgets.Layout(width='25%', margin='0 10px')),
    widgets.Box([date_range], layout=widgets.Layout(width='50%')),
])


### Zeile 2: "Übersicht"-Text & Geräte Auswahl
##############################################

geraete_filter_box = widgets.VBox([geraete_select])
geraete_filter_box.layout = widgets.Layout(width='50%', min_width='300px')

second_row = widgets.HBox([
    summary_output,
    geraete_filter_box,
    widgets.Box([metric_selector], layout=widgets.Layout(width='20%')),
], layout=widgets.Layout(justify_content='space-between', width='100%', padding='10px 0'))


# Zeile 3: Content bzw. Output! 
content_left = widgets.Output()
content_right = widgets.Output()
content = widgets.HBox([content_left, content_right])

content_left.layout = widgets.Layout(width='20%', padding='10px')
content_right.layout = widgets.Layout(width='80%', padding='10px')

content.layout = widgets.Layout(width='100%')




# Controls in VBox
controls = widgets.VBox([
    controls_top,
    second_row,
    content
], layout=widgets.Layout(width='100%', padding='10px'))



# Aufruf der Funktion, Dass Geräte Auswahl immer geupdatet werden soll
update_geraete_options(person_dropdown.value, gym_dropdown.value, date_range.value)


# Initial die Funktion aufrufen, für direkte Anzeige
update_dashboard(
    person_dropdown.value,
    gym_dropdown.value,
    geraete_select.value,
    date_range.value
)


# Alle Widgets beobachten auf 'value' Änderung
for w in [person_dropdown, gym_dropdown, geraete_select, date_range, metric_selector]:
    w.observe(on_change, names='value')


display(controls)